# JS04 Regression - Lab 0
## Creating Training, Validation, and Test Data

Dataset: Titanic feature data with the `Survived` label.

## Random Split

The data is split into training, validation, and test sets with an 8:1:1 ratio.

In [1]:
import pandas as pd
from sklearn.model_selection import KFold, train_test_split

df = pd.read_csv('Titanic-Dataset-selected.csv')
df.head()

,Survived,Pclass,Age,Sex,Cabin
0,0,3,-0.592481,1,115
1,1,1,0.638789,0,81
2,1,3,-0.284663,0,115
3,1,1,0.407926,0,55
4,0,3,0.407926,1,115


In [2]:
df_train, df_unseen = train_test_split(df, test_size=0.2, random_state=0)
df_val, df_test = train_test_split(df_unseen, test_size=0.5, random_state=0)

print(f'Original data count: {df.shape[0]}')
print(f'Training data count: {df_train.shape[0]}')
print(f'Validation data count: {df_val.shape[0]}')
print(f'Test data count: {df_test.shape[0]}')
print('=========')
print(f'Original label counts:\n{df.Survived.value_counts()}')
print(f'Training label counts:\n{df_train.Survived.value_counts()}')
print(f'Validation label counts:\n{df_val.Survived.value_counts()}')
print(f'Test label counts:\n{df_test.Survived.value_counts()}')

assert (len(df_train), len(df_val), len(df_test)) == (712, 89, 90)
assert len(set(df_train.index) & set(df_val.index)) == 0
assert len(set(df_train.index) & set(df_test.index)) == 0
assert len(set(df_val.index) & set(df_test.index)) == 0

Original data count: 891
Training data count: 712
Validation data count: 89
Test data count: 90
Original label counts:
Survived
0    549
1    342
Name: count, dtype: int64
Training label counts:
Survived
0    439
1    273
Name: count, dtype: int64
Validation label counts:
Survived
0    53
1    36
Name: count, dtype: int64
Test label counts:
Survived
0    57
1    33
Name: count, dtype: int64


## Stratified Split

Stratification preserves the proportion of the `Survived` labels in each split.

In [3]:
df2 = pd.read_csv('Titanic-Dataset-selected.csv')
df2_train, df2_unseen = train_test_split(
    df2, test_size=0.2, random_state=0, stratify=df2['Survived']
)
df2_val, df2_test = train_test_split(
    df2_unseen, test_size=0.5, random_state=0, stratify=df2_unseen['Survived']
)

print(f'Original label counts:\n{df2.Survived.value_counts()}')
print(f'Training label counts:\n{df2_train.Survived.value_counts()}')
print(f'Validation label counts:\n{df2_val.Survived.value_counts()}')
print(f'Test label counts:\n{df2_test.Survived.value_counts()}')

assert (len(df2_train), len(df2_val), len(df2_test)) == (712, 89, 90)
assert df2_train['Survived'].value_counts().to_dict() == {0: 439, 1: 273}
assert df2_val['Survived'].value_counts().to_dict() == {0: 55, 1: 34}
assert df2_test['Survived'].value_counts().to_dict() == {0: 55, 1: 35}

Original label counts:
Survived
0    549
1    342
Name: count, dtype: int64
Training label counts:
Survived
0    439
1    273
Name: count, dtype: int64
Validation label counts:
Survived
0    55
1    34
Name: count, dtype: int64
Test label counts:
Survived
0    55
1    35
Name: count, dtype: int64


## Cross Validation 1

K-Fold creates four training/test folds from the complete dataset.

In [4]:
df3 = pd.read_csv('Titanic-Dataset-selected.csv')
kf = KFold(n_splits=4)
folds = list(kf.split(df3))

print(f'Number of folds: {kf.get_n_splits()}')
print(f'KFold object: {kf}')
print(f'Number of rows in df3: {df3.shape[0]}')
for fold_number, (train_index, test_index) in enumerate(folds, start=1):
    print(f'Fold {fold_number}: train={len(train_index)}, test={len(test_index)}')
    assert len(set(train_index) & set(test_index)) == 0

all_test_indices = [index for _, test_index in folds for index in test_index]
assert sorted(all_test_indices) == list(range(len(df3)))

Number of folds: 4
KFold object: KFold(n_splits=4, random_state=None, shuffle=False)
Number of rows in df3: 891
Fold 1: train=668, test=223
Fold 2: train=668, test=223
Fold 3: train=668, test=223
Fold 4: train=669, test=222


## Cross Validation 2

First reserve 20% of the data for final testing, then apply four-fold cross-validation to the remaining 80% for training and validation.

In [5]:
df4 = pd.read_csv('Titanic-Dataset-selected.csv')
df4_train, df4_test = train_test_split(df4, test_size=0.2, random_state=0)
kf2 = KFold(n_splits=4)
folds2 = list(kf2.split(df4_train))

print(f'Number of folds: {kf2.get_n_splits()}')
print(f'KFold object: {kf2}')
print(f'Training rows: {df4_train.shape[0]}')
print(f'Test rows: {df4_test.shape[0]}')
for fold_number, (train_index, validation_index) in enumerate(folds2, start=1):
    print(f'Fold {fold_number}: train={len(train_index)}, validation={len(validation_index)}')
    assert len(set(train_index) & set(validation_index)) == 0

all_validation_indices = [index for _, validation_index in folds2 for index in validation_index]
assert sorted(all_validation_indices) == list(range(len(df4_train)))
assert len(set(df4_train.index) & set(df4_test.index)) == 0

Number of folds: 4
KFold object: KFold(n_splits=4, random_state=None, shuffle=False)
Training rows: 712
Test rows: 179
Fold 1: train=534, validation=178
Fold 2: train=534, validation=178
Fold 3: train=534, validation=178
Fold 4: train=534, validation=178


## Conclusion

Random splitting is simple, stratified splitting maintains label proportions, and cross-validation reuses the available training data across multiple folds while keeping the final test set separate.